# 第一课：Token 的奇妙旅程：从哪里来，到哪里去？

在这节课中，我们将深入大语言模型（LLM）的底层，探索 LLM 能够理解和生成的最小单位 —— **Token**。
我们会解答以下问题：
- **Token 是什么？** 大语言模型的工作模式是什么？
- **为什么我们需要分词方案（Tokenizer）？** BPE 算法是什么？
- **多模态模型中的 Token 是怎样的？** 视觉和声音如何被 Token 化？
- **Token 如何变成可计算的？** 探索 Embedding 层。
- **整个流转过程是怎样的？** 从 Token 输入到下一个 Token 输出的 Transformer 之旅。

## 1. 大语言模型的工作模式与 Token 的概念

大语言模型（如 ChatGPT, Qwen, Llama）的核心工作模式其实非常简单：**“给定一段上下文，预测下一个‘字’应该是什么”**。这种模式被称为**自回归（Autoregressive）生成**。

然而，这里说的“字”并不是我们日常理解中的汉字或英文字母，而是 **Token**。
Token 是模型处理信息的最小基本单位。它可能是一个词、一个字、甚至是一个词根。

让我们通过实际的代码来看看，一句话是如何被切分成 Token 的。

In [ ]:
# 如果没有安装依赖，请取消注释并运行下面这行
# !pip install transformers tiktoken torch matplotlib seaborn scikit-learn
# import os
# os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

from modelscope import AutoTokenizer, AutoModelForCausalLM

tokenizer_id = "Qwen/Qwen3.5-0.8B"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_id)

text = "Tokenization is awesome! 深度学习非常有趣。"
tokens = tokenizer.tokenize(text)
token_ids = tokenizer.encode(text)

print(f"原始文本: {text}")
print(f"\n--- 分词结果 ---")
for i, (tok, tid) in enumerate(zip(tokens, token_ids)):
    decoded = tokenizer.decode([tid])
    print(f"  [{i}] Token: {tok!r:20s}  ->  解码: {decoded!r:10s}  ->  ID: {tid}")
print(f"\nToken IDs: {token_ids}")

### 为什么 Token 和我们理解的“字”不一样？

观察上面的分词结果你会发现：
1. 英文中的 `Tokenization` 被切分成了 `Token` 和 `ization` 两个子词（Subword），这是因为 BPE 算法根据语料频率决定切分粒度。
2. 中文的“深度学习非常有趣”被切分成了若干个 Token，每个 Token 可能对应一个或多个汉字。
3. 你可能注意到 `tokenize()` 返回的中文 Token 看起来像乱码（如 `æ·±åº¦`），这是因为 **BPE 在 Byte 级别工作**：中文的 UTF-8 编码被拆解为原始字节后，用一种特殊的可见字符映射来表示。但当我们用 `decode()` 将 Token ID 解码回文本时，就能正常还原出中文。

**为什么不直接用字母或汉字？**
- **如果用字母（Character-level）**：英文只有 26 个字母，词表极小。但一句话会变成非常长的序列，模型需要看很远才能理解一个词的意思，计算效率极低，且难以捕捉语义。
- **如果用词（Word-level）**：“学习”、“学过”、“学了”会被当成完全不同的独立词汇，导致词表无限膨胀（Out-Of-Vocabulary 问题），且无法学习到词根之间的关联。

这就引入了现代大模型最常用的分词方案 —— **BPE (Byte-Pair Encoding)**。

## 2. 分词方案：BPE (Byte-Pair Encoding) 算法解析

BPE 是一种**数据驱动**的子词（Subword）分词算法。它的核心思想是：**找到语料库中最常连续出现的字符组合，把它们合并成一个新的 Token。**

- **对于英文**：像 "ing", "ly", "tion" 这样的常见词缀会被合并成一个独立的 Token，而生僻词会被拆解成常见的词根。
- **对于中文**：因为 UTF-8 编码下一个汉字占 3 个 Byte。很多基于 BPE 的模型是在 Byte 级别进行合并的（BBPE）。常见的汉字组合（如“中国”、“深度”）会成为独立的 Token，而生僻字可能会被拆解为底层的 Byte 组合。

这也是为什么 `tokenize()` 方法对中文返回的 Token 看起来像乱码 —— 它展示的是 Byte 级别的内部表示。但模型内部并不直接使用这些字符串，而是使用 **Token ID**（整数索引）。通过 `decode()` 方法，我们可以从 Token ID 完美还原出原始文本。

### 探索词表：模型“认识”多少个 Token？

每个 Tokenizer 都有一个固定的**词表（Vocabulary）**，里面存储了所有它“认识”的 Token。让我们来看看 Qwen3.5 的词表长什么样。

In [ ]:
print(f"模型: {tokenizer_id}")
print(f"词表大小 (vocab_size): {len(tokenizer)}")
print(f"模型最大输入长度: {tokenizer.model_max_length}")
print(f"\n--- 特殊 Token ---")
special_tokens = tokenizer.special_tokens_map
for name, value in special_tokens.items():
    token_id = tokenizer.convert_tokens_to_ids(value)
    print(f"  {name:15s} -> {value!r:20s}  (ID: {token_id})")

print(f"\n--- 词表中的 Token 示例 ---")
examples = ["hello", "世界", "ing", "Ġthe", "Ġis", "</w>"]
for ex in examples:
    tid = tokenizer.convert_tokens_to_ids(ex)
    in_vocab = tid != tokenizer.unk_token_id
    status = f"ID={tid}" if in_vocab else f"不在词表中 (返回 UNK ID={tid})"
    print(f"  {ex!r:20s} -> {status}")

print(f"\n--- 词表 ID 范围 ---")
print(f"  最小 ID: 0, 最大 ID: {len(tokenizer) - 1}")
print(f"\n--- 词表首尾 Token ---")
for i in [0, 1, 2, 3, 4]:
    print(f"  ID {i}: {tokenizer.decode([i])!r}")
print(f"  ...")
for i in range(len(tokenizer) - 3, len(tokenizer)):
    print(f"  ID {i}: {tokenizer.decode([i])!r}")

### 中英文分词差异对比

让我们直观地对比一下，同一段文本中中文和英文的分词粒度差异。

In [ ]:
en_text = "Natural language processing is fascinating."
zh_text = "自然语言处理非常有趣。"

for label, txt in [("英文", en_text), ("中文", zh_text)]:
    ids = tokenizer.encode(txt)
    decoded_tokens = [tokenizer.decode([tid]) for tid in ids]
    print(f"=== {label} ===")
    print(f"原文: {txt}")
    print(f"Token 数量: {len(ids)}")
    print(f"逐 Token 解码: {decoded_tokens}")
    print()

你会发现，同样语义的句子，中文通常比英文消耗更多的 Token。这是因为中文的 UTF-8 编码每个字占 3 个字节，BPE 合并的效率相对较低。这也是为什么在使用大模型 API 时，**中文的 Token 消耗（和费用）通常比英文更高**。

## 3. 多模态的 Token 是什么？

在当前的大模型时代，不仅文本可以被 Token 化，图像和声音也可以！**“Everything is a Token”** 是当前多模态大模型能够统一处理不同模态数据的核心所在。

- **视觉 Token (Vision)**：通常通过 ViT (Vision Transformer) 将一张图片切分成多个网格（Patches）。例如一张 $224 \times 224$ 的图片切成 $16 \times 16$ 的小块。每个小块展平并经过线性映射后，就变成了一个视觉 Token，其地位和文本 Token 毫无区别。
- **声音 Token (Audio)**：将连续的音频波形转化为频谱图，或者使用基于量化的声学模型（如 HuBERT, EnCodec）将音频片段映射为离散的 Codebook 索引，从而变成音频 Token。

## 4. Token 怎么变成可计算的？(Embedding 可视化)

神经网络无法直接计算字符串（如 "Token" 或 "学习"）或整数 ID（如 1234）。我们需要将离散的 Token 映射为连续的稠密向量，这就是 **Embedding（词嵌入）**。

这些 Embedding 并不是随机瞎猜的，而是**跟随 LLM 的整个训练过程一起学习得到的**。意思相近或语法作用相似的 Token，它们在高维空间中的向量距离也会更近。

我们来加载 Qwen3.5-0.8B 模型，提取它的 Embedding 层（即第一层），并直观地看一看。

In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import logging

logging.getLogger("transformers").setLevel(logging.ERROR)

model = AutoModelForCausalLM.from_pretrained(tokenizer_id)

embedding_layer = model.get_input_embeddings()
print(f"Embedding 层结构: {embedding_layer}")

words = ["king", "queen", "man", "woman", "苹果", "香蕉", "汽车"]
word_ids = [tokenizer.encode(w)[0] for w in words]

with torch.no_grad():
    word_embeddings = embedding_layer(torch.tensor(word_ids))

print(f"\n提取的向量矩阵形状: {word_embeddings.shape} -> 代表 {len(words)} 个词，每个词 {word_embeddings.shape[1]} 维")

plt.figure(figsize=(12, 5))
sns.heatmap(word_embeddings[:, :50].numpy(), cmap="coolwarm", yticklabels=words)
plt.title("Token Embeddings Heatmap (First 50 Dimensions)")
plt.xlabel("Embedding Dimensions")
plt.ylabel("Tokens")

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False
plt.show()

## 5. 整个流转过程：从 Token 到预测结果的“下一个 Token”

一个 Token 进入模型后，经历了怎样的奇幻漂流？整个流水线如下：

1. **输入阶段**：输入文本 $\rightarrow$ Tokenize 得到 `[ID_1, ID_2, ...]` $\rightarrow$ 查表得到 **Embedding 矩阵**。
2. **Transformer 层 (多层变换)**：
    - 向量矩阵进入几十层叠加的 Transformer Block。
    - **自注意力机制 (Self-Attention)**：这也是一种“线性+Softmax”操作，向量们开始互相“交流”。每个 Token 根据上下文，吸收其他 Token 的信息，更新自己的表示。
    - **前馈神经网络 (MLP/FFN)**：对更新后的向量进行线性映射 + 非线性激活函数（如 SwiGLU），完成高维空间的特征提取。
3. **输出阶段**：
    - 最后一层的隐藏状态向量（Hidden States）通过一个线性层（通常叫做 `lm_head`），将其**映射回词表大小的维度**。
    - 这些维度的值代表了预测每一个可能词汇的**原始得分 (Logits)**。
    - 经过 **Softmax** 函数，将得分转化为概率（总和为 1）。
    - 选出概率最高（或通过 Top-K 采样）的 Token ID，作为预测的下一个 Token。

让我们用代码把这个黑盒拆解，单步执行一遍前向传播（Forward Pass）！

In [ ]:
import torch.nn.functional as F

input_text = "人工智能是未来的"
print(f"输入文本: '{input_text}'")

inputs = tokenizer(input_text, return_tensors="pt")
input_ids = inputs["input_ids"]
print(f"\n[Step 1] Token IDs: {input_ids.tolist()[0]}")

with torch.no_grad():
    embeddings = model.model.embed_tokens(input_ids)
print(f"[Step 2] Embedding 形状: {embeddings.shape} -> (batch_size, seq_len, hidden_size)")

with torch.no_grad():
    transformer_outputs = model.model(input_ids)
    hidden_states = transformer_outputs.last_hidden_state
print(f"[Step 3] 最后一层 Hidden States 形状: {hidden_states.shape}")

last_token_hidden_state = hidden_states[:, -1, :] 
print(f"[Step 4] 用于预测的最后一个 Token 的向量形状: {last_token_hidden_state.shape}")

with torch.no_grad():
    logits = model.lm_head(last_token_hidden_state)
print(f"[Step 5] Logits 形状: {logits.shape} -> (batch_size, vocab_size={len(tokenizer)})")

probs = F.softmax(logits, dim=-1)
next_token_id = torch.argmax(probs, dim=-1).item()
next_token_str = tokenizer.decode([next_token_id])

print(f"\n=== 预测结果 ===")
print(f"最高概率的 Token ID: {next_token_id}")
print(f"对应的文本是: '{next_token_str}'")
print(f"拼接后的完整句子: '{input_text + next_token_str}'")

## 结语

通过这节课，我们梳理了 Token 的前世今生：

从人类可读的**文本/图像/音频** $\rightarrow$ 切分并映射为模型可消化的离散 **Token ID** $\rightarrow$ 转化为高维连续空间的 **Embedding** $\rightarrow$ 在 **Transformer** 的汪洋大海中进行无数次线性和非线性变换 $\rightarrow$ 最终收束为词表大小的 **概率分布** $\rightarrow$ 吐出下一个 **Token**。

这就是目前所有主流大语言模型（GPT, Claude, Llama, Qwen 等）共同的底层心跳节奏！